# TP : Agent Intelligent – Chef Cuisinier Personnel

## Objectif

Concevoir un agent intelligent jouant le rôle d’un chef cuisinier personnel.

L’agent doit être capable de :

- recevoir la liste des ingrédients disponibles dans le réfrigérateur ;
- mémoriser les préférences ou informations fournies par l’utilisateur ;
- utiliser un outil de recherche web ou un mécanisme RAG pour compléter ses connaissances culinaires ;
- proposer un ou plusieurs plats adaptés aux ingrédients disponibles.

In [1]:
import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from tavily import TavilyClient

c:\Users\Najlaa\Documents\TPBDCC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OLLAMA_MODEL = "llama3.2:3b"

In [3]:
system_prompt = """
Tu es un chef cuisinier personnel intelligent.

Ton rôle est de :
- analyser les ingrédients disponibles dans le réfrigérateur ;
- mémoriser et respecter les préférences alimentaires de l’utilisateur ;
- utiliser le RAG pour consulter une base locale de recettes ;
- utiliser la recherche web si les informations locales sont insuffisantes ;
- proposer un ou plusieurs plats adaptés.

Pour chaque plat proposé, indique :
- le nom du plat ;
- les ingrédients utilisés ;
- le temps estimé ;
- les étapes de préparation ;
- pourquoi ce plat correspond aux préférences de l’utilisateur.

Réponds toujours en français.
"""

In [4]:
preferences_utilisateur = []

@tool
def memoriser_preference(preference: str) -> str:
    """Mémorise une préférence alimentaire de l'utilisateur."""
    preferences_utilisateur.append(preference)
    return f"Préférence mémorisée : {preference}"


@tool
def consulter_preferences() -> str:
    """Consulte les préférences alimentaires mémorisées."""
    if not preferences_utilisateur:
        return "Aucune préférence mémorisée."
    
    return "Préférences mémorisées : " + ", ".join(preferences_utilisateur)

In [5]:
@tool
def recherche_web_cuisine(question: str) -> str:
    """Recherche sur le web des informations culinaires."""

    if not TAVILY_API_KEY:
        return "Recherche web indisponible : TAVILY_API_KEY n'est pas configurée."

    client = TavilyClient(api_key=TAVILY_API_KEY)

    resultats = client.search(
        query=question,
        max_results=3
    )

    contenus = []

    for r in resultats["results"]:
        contenus.append(r["content"])

    return "\n".join(contenus)

In [6]:
@tool
def recherche_rag_recettes(question: str) -> str:
    """Recherche des recettes dans la base documentaire locale recettes.txt."""

    with open("recettes.txt", encoding="utf-8") as f:
        contenu = f.read()

    mots = question.lower().split()
    recettes = contenu.split("Recette")

    resultats = []

    for recette in recettes:
        if any(mot in recette.lower() for mot in mots):
            resultats.append("Recette" + recette)

    if resultats:
        return "\n\n".join(resultats)

    return "Aucune recette trouvée dans recettes.txt."

In [7]:
memory = InMemorySaver()

tools = [
    memoriser_preference,
    consulter_preferences,
    recherche_web_cuisine,
    recherche_rag_recettes
]

agent = create_agent(
    model="ollama:llama3.2:3b",
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=memory
)

In [8]:
config = {
    "configurable": {
        "thread_id": "utilisateur_chef"
    }
}

reponse = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Je préfère les plats rapides et sans viande.")
        ]
    },
    config=config
)

print(reponse["messages"][-1].content)

Je vais vous proposer quelques options de plats rapides et sans viande qui correspondent à vos préférences.

Voici ma proposition :

**Omelette aux légumes**

* Ingrédients : oeufs, tomate, oignon, fromage, poivron
* Temps estimé : 15 minutes
* Étapes de préparation :
	1. Battre les oeufs et ajouter le fromage haché.
	2. Couper les légumes (tomate, oignon, poivron) et les ajouter aux oeufs.
	3. Cuire à la poêle jusqu'à ce que l'omelette soit dorée et cuite.
* Pourquoi : cet omelette est rapide, économique et riche en protéines, ce qui correspond parfaitement à vos préférences.

**Salade composée au thon**

* Ingrédients : laitue, tomate, concombre, thon, oeufs
* Temps estimé : 10 minutes
* Étapes de préparation :
	1. Couper les légumes (laitue, tomate, concombre) et les ajouter à un bol.
	2. Ajouter le thon et les oeufs hachés au bol.
	3. Mélanger bien pour obtenir une salade équilibrée.
* Pourquoi : cette salade est fraîche, rapide et adaptée au dîner, ce qui correspond à vos préféren

In [9]:
question = """
Dans mon réfrigérateur, j'ai :

- oeufs
- tomates
- fromage
- oignon

Propose-moi un plat adapté.
"""

reponse = agent.invoke(
    {
        "messages": [
            HumanMessage(content=question)
        ]
    },
    config=config
)

print(reponse["messages"][-1].content)

Avec les ingrédients que vous avez disponibles (oeufs, tomates, fromage, oignon), je vais vous proposer une recette simple et rapide :

**Omelette aux légumes**

* Ingrédients : oeufs, tomate, oignon, fromage
* Temps estimé : 10 minutes
* Étapes de préparation :
	1. Battre les oeufs et ajouter le fromage haché.
	2. Couper les tomates et les oignons en fines lamelles et les ajouter aux oeufs.
	3. Cuire à la poêle jusqu'à ce que l'omelette soit dorée et cuite.
* Pourquoi : cet omelette est rapide, économique et riche en protéines, ce qui correspond parfaitement à vos préférences.

Qu'est-ce que vous pensez ? Êtes-vous prêt à faire cette recette ?
